[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyneuro/Chapter_Colabs/blob/main/Colab_F.ipynb)

# Set F — Small Networks: STM & WTA (LEGO–Colab)
**Author:** Neural Engineering Laboratory, University of Missouri-
Gregory Glickert, Khuram Choudhry, Ziao Chen, Satish S. Nair

---

## 📖 Variable Key & Roadmap
<details>
<summary><b>Click to expand: Variable Key & Module Roadmap</b></summary>

### **The "Levers" (Network Parameters)**
| Variable | Definition | Unit | Instructional Role |
| :--- | :--- | :--- | :--- |
| `weight_EE` | Excitatory to Excitatory strength | μS | Controls "Reverberation" / Persistence duration |
| `weight_EI` | Excitatory to Inhibitory strength | μS | Controls how fast inhibition is recruited |
| `weight_IE` | Inhibitory to Excitatory strength | μS | Controls competitive suppression (Winner-Take-All) |
| `tau_syn` | Synaptic decay constant | ms | Determines the "temporal integration" of the network |
| `delay` | Synaptic delay | ms | Affects synchrony and decision stability |

### **The Roadmap**
* **F0 Starter:** Initializing the multi-cell "Engine" and helper builders.
* **F1 Excitatory STM:** Can a brief pulse trigger a memory that outlasts the stimulus?
* **F2 Inhibitory STM:** Using inhibition to "sculpt" or quench persistence.
* **F3 WTA Baseline:** What happens when cells compete without a "referee" (inhibition)?
* **F4 Global WTA:** Implementing the "Hard WTA" mechanism for crisp decisions.
* **F5 Contrast vs. Speed:** Quantifying how "easy" vs "hard" decisions affect reaction time.

</details>

In [45]:
# @title F0: Initialize Environment (High-Excitability Version) { display-mode: "form" }
!pip -q install neuron==8.2.4 >/dev/null 2>&1
try: from neuron import h, gui
except Exception: from neuron import h
from neuron.units import ms, mV
import numpy as np
import matplotlib.pyplot as plt

h.load_file('stdrun.hoc')

def build_population(n, name_prefix='E'):
    cells = []
    for i in range(n):
        sec = h.Section(name=f'{name_prefix}_{i}')
        sec.L = sec.diam = 20
        sec.insert('hh')
        for seg in sec:
            # Tuned for easier ignition:
            seg.hh.gnabar = 0.20  # Increased Sodium
            seg.hh.gkbar = 0.03   # Decreased Potassium
            seg.hh.gl = 0.0001    # Half the standard leak
            seg.hh.el = -65       # Standard resting potential
        cells.append(sec)
    return cells

def connect_all_to_all(cells, kind='E', tau=5.0, weight=0.01, delay=1.0, e_rev=0.0):
    syns = []
    ncs = []
    if kind == 'I': e_rev = -75.0

    # We apply a 20x boost to the weight to ensure the sliders are effective
    boosted_w = weight * 20

    for post in cells:
        for pre in cells:
            if pre == post: continue
            syn = h.ExpSyn(post(0.5))
            syn.tau = tau
            syn.e = e_rev
            nc = h.NetCon(pre(0.5)._ref_v, syn, sec=pre)
            nc.weight[0] = boosted_w
            nc.delay = delay
            nc.threshold = -20
            syns.append(syn)
            ncs.append(nc)
    return syns, ncs

def brief_iclamp(section, delay=5, dur=2, amp=0.5):
    stim = h.IClamp(section(0.5))
    stim.delay = delay; stim.dur = dur; stim.amp = amp
    return stim

def mk_rec(sec):
    return h.Vector().record(sec(0.5)._ref_v)

print("F0 ready. Memory cleared. Proceed to F1.")

F0 ready. Memory cleared. Proceed to F1.


### **F1 — STM (excitatory all-to-all)**

**Idea**: A brief pulse to one excitatory unit enters a recurrent $E \to E$ network. Sufficient weight and excitatory $\tau$ produce post-stimulus persistence, a hallmark of Short-Term Memory (STM).

**What to change**  
*   **Weight ($E \to E$):** Approximately 0.004–0.010.
*   **$\tau$ (E synaptic decay):** 1–5 ms.
*   **Delay ($E \to E$):** 0.5–2 ms.
*   **Stimulus:** Pulse amplitude and duration in `brief_iclamp`.

**Sanity checks**  
1.  Observe if spiking continues after the external pulse ends.
2.  Report the **persistence duration** (time from pulse offset to the last recorded spike).

**Predict → verify**  
*   **Persistence:** Increasing weight or $\tau$ should lengthen persistence until it reaches a saturation regime.
*   **Stability:** Excessive values may yield synchronized or runaway spiking.

**Exercises**  
1.  Find a (weight, $\tau$) pair that yields **50–150 ms** of persistence; capture the raster/trace and record the duration.
2.  Double the membrane leak ($g_{pas}$) and re-measure; explain the change using RC integrator logic.
3.  Identify a parameter set that fails to sustain activity; state which specific lever is currently limiting the network.

---



In [52]:
# @title F1: STM Experimental Rig (Dashboard + Solution Logger) { display-mode: "form" }
from ipywidgets import interactive_output, HBox, VBox, FloatSlider, Layout, Label
import matplotlib.pyplot as plt

def simulate_f1_final(w_ee, tau_e, delay, stim_amp, stim_dur):
    # Reset NEURON state
    h('forall delete_section()')
    N = 5
    cells = build_population(N, name_prefix='E')

    # 1. Connect network (using the 20x boost from F0)
    syns, ncs = connect_all_to_all(cells, kind='E', tau=tau_e, weight=w_ee, delay=delay)

    # 2. Apply stimulus to Cell 0
    stim = brief_iclamp(cells[0], delay=5, dur=stim_dur, amp=stim_amp)

    # 3. Setup recording
    t = h.Vector().record(h._ref_t)
    recs = [mk_rec(c) for c in cells]
    spikes = [h.Vector() for _ in range(N)]
    for i in range(N):
        nc = h.NetCon(cells[i](0.5)._ref_v, None, sec=cells[i])
        nc.threshold = -20
        nc.record(spikes[i])

    # 4. Run Simulation
    h.finitialize(-65 * mV)
    h.continuerun(250 * ms)

    # --- Analysis & Success Logic ---
    total_spikes = sum([len(s) for s in spikes])
    is_ignited = total_spikes > 15

    # 5. Visualization
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6), sharex=True,
                                   gridspec_kw={'height_ratios': [3, 1]})
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']

    for i in range(N):
        ax1.plot(np.array(t), np.array(recs[i]) + (i*3), color=colors[i])
        ax2.vlines(np.array(spikes[i]), i + 0.6, i + 1.4, color=colors[i], linewidth=2)

    status = "SUCCESS: Ignited" if is_ignited else "FAILED: No Persistence"
    ax1.set_ylabel('V (mV) + Offset')
    ax1.set_title(f'{status} (Total Spikes: {total_spikes})')
    ax2.axvspan(5, 5 + stim_dur, color='gray', alpha=0.2)
    ax2.set_xlabel('Time (ms)')
    plt.tight_layout()
    plt.show()

    if is_ignited:
        print(f"--- SOLUTION PRESERVED ---")
        print(f"Values: W_EE={w_ee:.3f}, Tau_E={tau_e:.3f}, Delay={delay:.3f}")
        print(f"Copy for Report: [STM_SUCCESS: {total_spikes} spikes]")

# --- UI Setup ---
# All sliders now have 0.001 resolution (3 decimals)
# Defaults set to your successful 'Ignition' values
s_lay = Layout(width='120px', height='180px')
style = {'description_width': 'initial'}

w_s  = FloatSlider(value=0.046, min=0.001, max=0.050, step=0.001,
                   description='W_EE', orientation='vertical', layout=s_lay, style=style, readout_format='.3f')
t_s  = FloatSlider(value=12.000, min=1.000, max=30.000, step=0.001,
                   description='Tau_E', orientation='vertical', layout=s_lay, style=style, readout_format='.3f')
d_s  = FloatSlider(value=0.200, min=0.100, max=5.000, step=0.001,
                   description='Delay', orientation='vertical', layout=s_lay, style=style, readout_format='.3f')
sa_s = FloatSlider(value=1.500, min=0.100, max=5.000, step=0.001,
                   description='Stim_A', orientation='vertical', layout=s_lay, style=style, readout_format='.3f')
sd_s = FloatSlider(value=10.000, min=1.000, max=50.000, step=0.001,
                   description='Stim_D', orientation='vertical', layout=s_lay, style=style, readout_format='.3f')

ui = HBox([w_s, t_s, d_s, sa_s, sd_s])
out = interactive_output(simulate_f1_final, {
    'w_ee': w_s, 'tau_e': t_s, 'delay': d_s, 'stim_amp': sa_s, 'stim_dur': sd_s
})

display(VBox([ui, out]))

### **F2 — STM (inhibitory all-to-all)**

**Idea**  
Introducing global inhibition (negative $E_{rev}$, longer $\tau_i$) shortens or stabilizes reverberation. The timing and gain determine whether persistence is truncated or fails to ignite.

**What to change**  
*   **$E_{rev}$:** Set to −75 mV (default).
*   **$\tau$ (I synapse):** 6–12 ms.
*   **Weight (I connections):** Approximately 0.006–0.020.

**Sanity checks**  
1.  Compare the **time to quench** with the results from F1.
2.  Verify that a brief "rebound" is possible without entering a sustained loop.

**Predict → verify**  
*   **Suppression:** Stronger inhibition or earlier arrival reduces persistence; excessive strength prevents ignition.
*   **Oscillation:** Longer $\tau_i$ can introduce oscillatory tails if synaptic delays are long.

**Exercises**  
1.  Tune parameters to produce a **single rebound** spike without sustained activity; include the traces.
2.  Sweep $\tau_i$ with $E$ fixed; plot **persistence duration vs. $\tau_i$**.
3.  Relate your observed changes to a biological mechanism (e.g., **shunting** vs. **hyperpolarizing** inhibition).

In [ ]:

N=5; cells_I=build_population(N)
synsI,ncsI=connect_all_to_all(cells_I,kind='I',tau=8.0,e_rev=-75.0,weight=0.01,delay=1.0)
drive0=brief_iclamp(cells_I[0],delay=5,dur=2,amp=0.4)
t,vm=run_and_record(cells_I,tstop=200,vlabels=[f'I[{i}]' for i in range(N)])
print('Increase inhibitory weight to quench activity faster.')


### **F3 — WTA baseline (no inhibition)**

**Idea**  
In a Winner-Take-All (WTA) scenario without shared inhibition, small differences in tonic input produce graded, overlapping outcomes. The "winner" is simply the cell with the largest drive.

**What to change**  
*   **Contrast ($\Delta amp$):** Use `linspace(0.08, 0.12, N)` to set tonic input amplitudes.
*   **Weight ($E \to E$):** 0.001–0.004.
*   **Runtime:** Ensure `tstop` is long enough to compute stable final means.

**Sanity checks**  
1.  Inspect the output of `time_to_win` (winner index, time, and mean rates).
2.  For very small $\Delta amp$, expect slow or undefined decisions.

**Predict → verify**  
*   **Ambiguity:** As $\Delta amp \to 0$, categories blur; without global inhibition, a "crisp" winner is rare.
*   **Exaggeration:** Larger $E \to E$ weights can exaggerate differences but risk network-wide synchrony.

**Exercises**  
1.  Plot the **winner index** across several $\Delta amp$ values and interpret the resulting ambiguity.
2.  Halve the $E \to E$ weight; determine if the winner changes and explain why.
3.  Justify why adding a shared inhibitory pool is necessary to achieve categorical WTA behavior.

In [ ]:

N=5; cells_WE=build_population(N)
synsWE,ncsWE=connect_all_to_all(cells_WE,kind='E',tau=2.0,e_rev=0.0,weight=0.003,delay=1.0)
amps=np.linspace(0.08,0.12,N)
ics=[tonic_iclamp(cells_WE[i],amp=float(amps[i]),delay=5.0) for i in range(N)]
t,vm=run_and_record(cells_WE,tstop=400,vlabels=[f'Eonly[{i}]' for i in range(N)])
w,tstar,means=time_to_win(t,vm)
print(f'Baseline (no inhibition): winner={w}, time_to_win≈{tstar}, means={means}')


### **F4 — WTA with global inhibition**

**Idea**  
Adding one inhibitory unit allows $E \to I$ recruitment to suppress all principals via $I \to E$ feedback. Proper gain and delay produce "hard" WTA decisions with a finite `time_to_win`.

**What to change**  
*   **Recruitment ($E \to I$ weight):** Approximately 0.004–0.010.
*   **Global Strength ($I \to E$ weight):** Approximately 0.01–0.04.
*   **$I \to E$ Delay:** 0.5–2.0 ms.
*   **Difficulty:** Keep $E \to E$ small and vary $\Delta amp$ to test task difficulty.

**Sanity checks**  
1.  Confirm a **single clear winner** and a finite `time_to_win`.
2.  If all units quench (stop firing), the $I \to E$ feedback is too strong or too fast.

**Predict → verify**  
*   **Transitions:** Increasing $I \to E$ transitions the network through: Coexistence $\to$ Soft WTA $\to$ Hard WTA $\to$ Quench.
*   **Dynamics:** Larger delays can cause units to "alternate" before one eventually settles as the winner.

**Exercises**  
1.  Sweep the $I \to E$ weight; plot **time_to_win vs. weight** at a fixed $\Delta amp$. Mark the minimal point required for Hard-WTA.
2.  Sweep the delay; show the **alternation regime** and explain the timing effects.
3.  Reduce $\Delta amp$ until the WTA mechanism fails; note the boundary (resembling a psychometric threshold).

---

In [ ]:

N=5; princ=build_population(N)
inh=h.Section(name='inh'); inh.L=inh.diam=18; inh.Ra=100; inh.cm=1; inh.insert('hh')
amps=np.linspace(0.08,0.12,N)
ics=[tonic_iclamp(princ[i],amp=float(amps[i]),delay=5.0) for i in range(N)]
E2I_syn=[]; E2I_nc=[]
for p in princ:
    syn=h.ExpSyn(inh(0.5)); syn.tau=2.0; syn.e=0.0
    nc=h.NetCon(p(0.5)._ref_v,syn,sec=p); nc.threshold=-20; nc.delay=1.0; nc.weight[0]=0.006
    E2I_syn.append(syn); E2I_nc.append(nc)
I2E_syn=[]; I2E_nc=[]
for p in princ:
    syn=h.ExpSyn(p(0.5)); syn.tau=8.0; syn.e=-75.0
    nc=h.NetCon(inh(0.5)._ref_v,syn,sec=inh); nc.threshold=-20; nc.delay=0.8; nc.weight[0]=0.02
    I2E_syn.append(syn); I2E_nc.append(nc)
_synSE,_ncSE=connect_all_to_all(princ,kind='E',tau=2.0,e_rev=0.0,weight=0.0015,delay=1.0)

t=h.Vector(); t.record(h._ref_t)
recP=[mk_rec(p) for p in princ]; recI=mk_rec(inh)

h.finitialize(-65*mV); h.continuerun(500*ms)
tnp=np.array(t); vP=[np.array(r) for r in recP]; vI=np.array(recI)
plot_traces(tnp,vP,[f'P[{i}]' for i in range(N)],title='F4: WTA with global inhibition')
w,tstar,means=time_to_win(tnp,vP)
print(f'Winner={w}, time_to_win≈{tstar}, final means={means}')


### **F5 — Contrast vs. decision time**

**Idea**  
Quantify the relationship between input contrast ($\Delta amp$) and `time_to_win` under Hard-WTA settings to generate a decision-time curve.

**What to change**  
*   **Contrasts:** Use a list like `[0.005, 0.01, 0.02, 0.03, 0.04]`.
*   **Base Amplitude:** Keep baseline drive low enough that cells remain below threshold without recurrent help.
*   **Inhibitory Weight:** Tune `inh_w` to maintain a Hard-WTA regime.

**Sanity checks**  
1.  Verify that `time_to_win` **decreases** as input contrast increases.
2.  Flat or increasing curves usually indicate a mismatch between recruitment and feedback.

**Predict → verify**  
*   **Noise:** Adding low-level noise will broaden the decision-time distribution near threshold contrasts.
*   **Speed:** Stronger $E \to I$ recruitment combined with balanced $I \to E$ feedback typically speeds decisions.

**Exercises**  
1.  Add small **Gaussian noise**; repeat 10 trials per contrast and plot **mean ± SD**.
2.  For a fixed contrast, vary the synaptic delay and report the effect on `time_to_win`.
3.  Rank $E \to I$, $I \to E$, and delay by their overall influence on **speed vs. stability**.

In [ ]:

import matplotlib.pyplot as plt

def wta_decision_time(contrast=0.04,base=0.08,N=5,inh_w=0.02):
    princ=build_population(N); inh=h.Section(name='inh2'); inh.L=inh.diam=18; inh.Ra=100; inh.cm=1; inh.insert('hh')
    amps=np.linspace(base,base+contrast,N)
    ics=[tonic_iclamp(princ[i],amp=float(amps[i]),delay=5.0) for i in range(N)]
    for p in princ:
        se=h.ExpSyn(inh(0.5)); se.tau=2.0; se.e=0.0
        nc=h.NetCon(p(0.5)._ref_v,se,sec=p); nc.threshold=-20; nc.delay=1.0; nc.weight[0]=0.006
    for p in princ:
        si=h.ExpSyn(p(0.5)); si.tau=8.0; si.e=-75.0
        nc=h.NetCon(inh(0.5)._ref_v,si,sec=inh); nc.threshold=-20; nc.delay=0.8; nc.weight[0]=inh_w
    t=h.Vector(); t.record(h._ref_t)
    recP=[mk_rec(p) for p in princ]
    h.finitialize(-65*mV); h.continuerun(500*ms)
    tnp=np.array(t); vP=[np.array(r) for r in recP]
    w,tstar,means=time_to_win(tnp,vP)
    return w,tstar,means

contrasts=[0.005,0.01,0.02,0.03,0.04]; times=[]
for c in contrasts:
    w,tstar,_=wta_decision_time(contrast=c)
    times.append(tstar)
plt.figure(figsize=(4.8,3)); plt.plot(contrasts,times,'o-')
plt.xlabel('Input contrast (Δamp)'); plt.ylabel('Time‑to‑win (ms)'); plt.title('F5: WTA decision time vs contrast')
plt.grid(True,alpha=0.3); plt.tight_layout(); plt.show()


## Reflection <a id='reflection'></a>
- Why does E‑E feedback support STM? How does leak counteract it?
- Which parameters most strongly shape WTA speed: E→I, I→E, Δinput, or τ?
- How could we add realistic noise and test robustness of decisions?


---

## Practice / Discussion Questions — Set F — Small Networks (STM & WTA)

1) Define **reverberation** and explain how it supports **persistence** in a small network.
2) Minimum conditions (connectivity + kinetics) that favor **STM** without runaway.
3) *Predict*: If NMDA‑like kinetics slow, how does persistence change? Why?
4) Describe a **WTA** network with a shared inhibitory pool; list two parameters controlling **time‑to‑win**.
5) How do small **tonic differences** become categorical outcomes in WTA?
6) Two **metrics** for persistence and one for WTA decision quality.
7) *Scenario*: Persistence collapses when inhibition increases slightly—two hypotheses.
8) Why is **inhibitory timing** still critical in networks?
9) Compare energy/robustness trade‑offs: **synaptic WM** vs **reverberation**.
10) *Design*: A perturbation to disambiguate persistent state vs noise.
11) What is an **attractor** here, and how would you show evidence for it?
12) One **failure mode** signaling need to adjust inhibitory balance.
13) How to test **stability** of a persistent state with parameter sweeps.
14) *Compare*: WTA vs soft‑WTA—what circuit change produces the difference?
15) Simple **readout** to quantify memory length as parameters vary.
16) Why can **adaptation** at single‑cell level influence persistence?
17) A **teaching visualization** to help students “see” WTA unfold.
18) One reason to teach both WM mechanisms (synaptic vs reverberation).
19) A **biologically plausible** way to end persistence without external reset.
20) How Set F positions you for **Set G** behavior mapping.
